In [3]:
# Install required libraries
# Run this once per Colab session. It installs tools we need.

!pip install yfinance pandas_datareader plotly --quiet

print("✅ Libraries installed successfully")


✅ Libraries installed successfully


In [4]:
# CELL 2: Import all libraries
# This tells Python: "get these tools ready for use"

import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
import warnings

from pandas_datareader import data as pdr
import yfinance as yf
from datetime import datetime

pd.options.display.float_format = "{:.4f}".format

print("✅ All imports successful")


✅ All imports successful


In [5]:
# CELL 3: Configuration - edit these if you want different dates or tenors
# These are the Federal Reserve's official codes for US Treasury yield series

START_DATE = "2015-01-01"
END_DATE = datetime.today().strftime("%Y-%m-%d")

# FRED series codes for each Treasury maturity
# e.g. DGS2 = 2-Year Treasury yield, DGS10 = 10-Year yield
FRED_TENOR_MAP = {
    "1M": "DGS1MO",
    "3M": "DGS3MO",
    "6M": "DGS6MO",
    "1Y":  "DGS1",
    "2Y":  "DGS2",
    "3Y":  "DGS3",
    "5Y":  "DGS5",
    "7Y":  "DGS7",
    "10Y": "DGS10",
    "20Y": "DGS20",
    "30Y": "DGS30",
}

# Maturity in years for each tenor label
TENOR_TO_YEARS = {
    "1M": 1/12, "3M": 3/12, "6M": 6/12,
    "1Y": 1.0,  "2Y": 2.0,  "3Y": 3.0,
    "5Y": 5.0,  "7Y": 7.0,  "10Y": 10.0,
    "20Y": 20.0,"30Y": 30.0
}

# ETF tickers used as hedge instruments
HEDGE_ETFS = ["SHY", "IEF", "TLT"]

# Approximate effective duration of each ETF (in years)
# SHY = short bonds, IEF = medium bonds, TLT = long bonds
ETF_APPROX_DURATIONS = {"SHY": 1.8, "IEF": 7.8, "TLT": 17.0}

print(f"✅ Config set: {START_DATE} to {END_DATE}")
print(f"   Tenors: {list(FRED_TENOR_MAP.keys())}")
print(f"   Hedge ETFs: {HEDGE_ETFS}")


✅ Config set: 2015-01-01 to 2026-06-19
   Tenors: ['1M', '3M', '6M', '1Y', '2Y', '3Y', '5Y', '7Y', '10Y', '20Y', '30Y']
   Hedge ETFs: ['SHY', 'IEF', 'TLT']


In [6]:
# CELL 4: Download Treasury yields from FRED (Federal Reserve Economic Data)
# This fetches daily yields for each maturity, all in one clean table.

all_series = []

for tenor, fred_code in FRED_TENOR_MAP.items():
    try:
        ser = pdr.DataReader(fred_code, "fred", START_DATE, END_DATE)
        ser = ser.rename(columns={fred_code: tenor})
        all_series.append(ser)
        print(f"  ✅ {tenor} ({fred_code}) — {len(ser)} rows fetched")
    except Exception as e:
        print(f"  ⚠️  Could not fetch {tenor} ({fred_code}): {e}")

# Combine all series into one wide table: rows = dates, columns = tenors
curve_df = pd.concat(all_series, axis=1)

# Convert from percent to decimal (e.g., 4.5% → 0.045)
curve_df = curve_df.astype(float) / 100.0

# Drop days where key tenors are missing
curve_df = curve_df.dropna(how="any", subset=["2Y", "5Y", "10Y", "30Y"])

print(f"\n✅ Curve table ready: {curve_df.shape[0]} dates × {curve_df.shape[1]} tenors")
print(curve_df.tail(3))


  ✅ 1M (DGS1MO) — 2990 rows fetched
  ✅ 3M (DGS3MO) — 2990 rows fetched
  ✅ 6M (DGS6MO) — 2990 rows fetched
  ✅ 1Y (DGS1) — 2990 rows fetched
  ✅ 2Y (DGS2) — 2990 rows fetched
  ✅ 3Y (DGS3) — 2990 rows fetched
  ✅ 5Y (DGS5) — 2990 rows fetched
  ✅ 7Y (DGS7) — 2990 rows fetched
  ✅ 10Y (DGS10) — 2990 rows fetched
  ✅ 20Y (DGS20) — 2990 rows fetched
  ✅ 30Y (DGS30) — 2990 rows fetched

✅ Curve table ready: 2866 dates × 11 tenors
               1M     3M     6M     1Y     2Y     3Y     5Y     7Y    10Y  \
DATE                                                                        
2026-06-15 0.0369 0.0379 0.0381 0.0384 0.0407 0.0410 0.0418 0.0432 0.0447   
2026-06-16 0.0367 0.0379 0.0381 0.0384 0.0405 0.0408 0.0416 0.0428 0.0443   
2026-06-17 0.0368 0.0383 0.0391 0.0398 0.0420 0.0423 0.0427 0.0437 0.0449   

              20Y    30Y  
DATE                      
2026-06-15 0.0497 0.0497  
2026-06-16 0.0492 0.0493  
2026-06-17 0.0495 0.0493  


In [7]:
# CELL 5: Download ETF prices from Yahoo Finance
# SHY = short-term Treasuries, IEF = medium-term, TLT = long-term

try:
    etf_data = yf.download(
        HEDGE_ETFS,
        start=START_DATE,
        end=END_DATE,
        auto_adjust=True,
        progress=False,
    )
    # Extract only the closing prices
    etf_prices_df = etf_data["Close"].copy()
    etf_prices_df = etf_prices_df.dropna(how="all")

    # Most recent prices
    latest_etf_prices = etf_prices_df.iloc[-1].to_dict()

    print("✅ ETF prices downloaded")
    print(f"   Rows: {len(etf_prices_df)}")
    print(f"   Latest prices: {latest_etf_prices}")

except Exception as e:
    print(f"❌ ETF download failed: {e}")


✅ ETF prices downloaded
   Rows: 2881
   Latest prices: {'IEF': 94.0199966430664, 'SHY': 81.87999725341797, 'TLT': 86.33000183105469}


In [8]:
# CELL 6: Visualise yield history for key tenors

tenors_to_plot = ["2Y", "5Y", "10Y", "30Y"]
df_plot = curve_df[tenors_to_plot].copy() * 100  # back to % for display

fig = px.line(
    df_plot,
    x=df_plot.index,
    y=df_plot.columns,
    title="US Treasury Constant-Maturity Yields (Source: FRED)",
    labels={"value": "Yield (%)", "variable": "Tenor", "index": "Date"},
)
fig.update_layout(template="plotly_white")
fig.show()

print("✅ Yield history chart displayed above")


✅ Yield history chart displayed above


In [10]:
# CELL 7: Bootstrap zero-coupon curve from par yields on a chosen date
# Par yields come from FRED. Zero rates are implied by stripping coupons out.

def bootstrap_zero_curve(par_curve: pd.Series) -> pd.DataFrame:
    """
    Sequential bootstrap: for each maturity, use prior discount factors
    to solve for the next zero rate. Annual coupon, face=100.
    """
    # Only use tenors we know the year value for
    tenors = [t for t in par_curve.index if t in TENOR_TO_YEARS]
    tenors = sorted(tenors, key=lambda t: TENOR_TO_YEARS[t])

    maturities_float = [TENOR_TO_YEARS[t] for t in tenors]
    par_yields = [float(par_curve[t]) for t in tenors]

    # Store calculated discount factors, keyed by integer maturity
    calculated_dfs = {}

    # Store results for the final DataFrame
    output_tenors = []
    output_maturities = []
    output_par_yields = []
    output_zero_rates = []
    output_discount_factors = []

    for idx, (T_float, R) in enumerate(zip(maturities_float, par_yields)):
        T_int = int(round(T_float)) # Integer part of maturity

        output_tenors.append(tenors[idx])
        output_maturities.append(T_float)
        output_par_yields.append(R * 100) # Store par yield in percent

        if T_int <= 0:
            d = np.nan
            s = np.nan
        elif T_int == 1:
            # Simple: price = (coupon + face) / (1 + zero_rate)
            d = 1.0 / (1.0 + R)
            s = R
        else:
            # Use prior discount factors for intermediate coupons
            coupon = R * 100 # annual cash coupon on 100 face

            # Sum of discounted intermediate coupon payments
            # We need discount factors for year 1, 2, ..., T_int - 1
            A = 0.0
            for year_k in range(1, T_int):
                if year_k in calculated_dfs:
                    A += coupon * calculated_dfs[year_k]
                else:
                    # If a prior discount factor is missing, we can't bootstrap
                    # This implies either problem with input data or algorithm assumption
                    # For now, we'll mark this as nan
                    A = np.nan
                    break # Cannot proceed if prior DF is missing

            if np.isnan(A):
                d = np.nan
                s = np.nan
            else:
                # Solve for the current discount factor
                # Bond Price = C*d(1) + C*d(2) + ... + C*d(T-1) + (C+Face)*d(T)
                # Here Face = 100, C = R*100, Bond Price = 100 (par)
                # 100 = A + (coupon + 100) * d
                d = (100 - A) / (coupon + 100)
                if d <= 0 or (coupon + 100) == 0: # Avoid division by zero and non-positive discount factors
                    d = np.nan
                    s = np.nan
                else:
                    # Convert discount factor to zero rate
                    # d = 1 / (1 + s)^T
                    # (1 + s)^T = 1 / d
                    # 1 + s = (1 / d)^(1/T)
                    # s = (1 / d)^(1/T) - 1
                    # Note: T_float is used here for accuracy, T_int for loop
                    s = (1.0 / d)**(1.0 / T_float) - 1.0


        # Store the calculated discount factor and zero rate
        output_discount_factors.append(d)
        output_zero_rates.append(s * 100 if not np.isnan(s) else np.nan) # Store zero rate in percent

        # Only store valid discount factors keyed by integer maturity
        # If T_int=1, and we have multiple tenors that round to 1 year (e.g., 6M and 1Y),
        # the last one (1Y) will overwrite the earlier ones, which is generally desired
        # for annual bootstrap. We only store if d is not NaN and T_int is positive (a valid year).
        if not np.isnan(d) and T_int > 0:
            calculated_dfs[T_int] = d

    return pd.DataFrame({
        "tenor": output_tenors,
        "maturity_years": output_maturities,
        "par_yield_pct": output_par_yields,
        "zero_rate_pct": output_zero_rates,
        "discount_factor": output_discount_factors,
    })

# Pick the most recent date available
as_of_date = curve_df.index.max()
par_curve = curve_df.loc[as_of_date].dropna()

zero_curve_df = bootstrap_zero_curve(par_curve)

print(f"✅ Zero curve bootstrapped for {as_of_date.date()}")
display(zero_curve_df)


✅ Zero curve bootstrapped for 2026-06-17


,tenor,maturity_years,par_yield_pct,zero_rate_pct,discount_factor
0,1M,0.0833,3.6800,NaN,NaN
1,3M,0.2500,3.8300,NaN,NaN
2,6M,0.5000,3.9100,NaN,NaN
3,1Y,1.0000,3.9800,3.9800,0.9617
4,2Y,2.0000,4.2000,4.2046,0.9209
5,3Y,3.0000,4.2300,4.2344,0.8830
6,5Y,5.0000,4.2700,NaN,NaN
7,7Y,7.0000,4.3700,NaN,NaN
8,10Y,10.0000,4.4900,NaN,NaN
9,20Y,20.0000,4.9500,NaN,NaN


In [11]:
# CELL 8: Chart - par yield vs bootstrapped zero rate

df_valid = zero_curve_df.dropna()

fig = go.Figure()
fig.add_trace(go.Scatter(
    x=df_valid["maturity_years"], y=df_valid["par_yield_pct"],
    mode="lines+markers", name="Par Yield"
))
fig.add_trace(go.Scatter(
    x=df_valid["maturity_years"], y=df_valid["zero_rate_pct"],
    mode="lines+markers", name="Zero Rate (Bootstrapped)"
))
fig.update_layout(
    title=f"Par vs Zero Curve — {as_of_date.date()}",
    xaxis_title="Maturity (years)",
    yaxis_title="Rate (%)",
    template="plotly_white"
)
fig.show()
print("✅ Par vs zero curve chart displayed")


✅ Par vs zero curve chart displayed


In [12]:
# CELL 9: Bond pricing, duration, DV01, and convexity functions

def price_bond(face, coupon_rate, ytm, maturity_years, freq=1):
    """Price a bond from yield to maturity."""
    n = int(maturity_years * freq)
    coupon = coupon_rate * face / freq
    y = ytm / freq
    times = np.arange(1, n + 1)
    cashflows = np.full(n, coupon)
    cashflows[-1] += face
    discounts = (1 + y) ** times
    return float(np.sum(cashflows / discounts))

def macaulay_duration(face, coupon_rate, ytm, maturity_years, freq=1):
    """Macaulay duration in years."""
    n = int(maturity_years * freq)
    coupon = coupon_rate * face / freq
    y = ytm / freq
    times = np.arange(1, n + 1) / freq
    cashflows = np.full(n, coupon)
    cashflows[-1] += face
    discounts = (1 + y) ** (times * freq)
    price = float(np.sum(cashflows / discounts))
    return float(np.sum(times * cashflows / discounts) / price)

def modified_duration(face, coupon_rate, ytm, maturity_years, freq=1):
    """Modified duration = Macaulay duration / (1 + y/freq)."""
    D_mac = macaulay_duration(face, coupon_rate, ytm, maturity_years, freq)
    return D_mac / (1 + ytm / freq)

def calc_dv01(face, coupon_rate, ytm, maturity_years, freq=1):
    """DV01: price change for 1 basis point move in yield."""
    price = price_bond(face, coupon_rate, ytm, maturity_years, freq)
    D_mod = modified_duration(face, coupon_rate, ytm, maturity_years, freq)
    return D_mod * price * 1e-4

def calc_convexity(face, coupon_rate, ytm, maturity_years, freq=1):
    """Convexity: curvature correction beyond duration."""
    n = int(maturity_years * freq)
    coupon = coupon_rate * face / freq
    y = ytm / freq
    times = np.arange(1, n + 1)
    cashflows = np.full(n, coupon)
    cashflows[-1] += face
    discounts = (1 + y) ** times
    price = float(np.sum(cashflows / discounts))
    return float(np.sum(cashflows / discounts * times * (times + 1)) / (price * (1 + y) ** 2))

print("✅ Bond analytics functions defined")
print("\nQuick test — 10Y bond at 4.5% coupon and yield:")
face, c, ytm, T = 100, 0.045, 0.045, 10
print(f"  Price:     {price_bond(face, c, ytm, T):.4f}")
print(f"  Mac Dur:   {macaulay_duration(face, c, ytm, T):.4f} years")
print(f"  Mod Dur:   {modified_duration(face, c, ytm, T):.4f} years")
print(f"  DV01:      {calc_dv01(face, c, ytm, T):.4f}")
print(f"  Convexity: {calc_convexity(face, c, ytm, T):.4f}")


✅ Bond analytics functions defined

Quick test — 10Y bond at 4.5% coupon and yield:
  Price:     100.0000
  Mac Dur:   8.2688 years
  Mod Dur:   7.9127 years
  DV01:      0.0791
  Convexity: 77.8103


In [13]:
# CELL 10: Define a stylized bond portfolio using today's par yields

# Use the most recently fetched curve
y2  = float(curve_df.loc[as_of_date, "2Y"])
y5  = float(curve_df.loc[as_of_date, "5Y"])
y10 = float(curve_df.loc[as_of_date, "10Y"])

portfolio = [
    {"name": "2Y Bond",  "face": 100, "coupon": y2,  "ytm": y2,  "maturity": 2.0,  "notional": 5_000_000},
    {"name": "5Y Bond",  "face": 100, "coupon": y5,  "ytm": y5,  "maturity": 5.0,  "notional": 5_000_000},
    {"name": "10Y Bond", "face": 100, "coupon": y10, "ytm": y10, "maturity": 10.0, "notional": 5_000_000},
]

rows = []
total_dv01 = 0.0

for bond in portfolio:
    price  = price_bond(bond["face"], bond["coupon"], bond["ytm"], bond["maturity"])
    dv01_  = calc_dv01(bond["face"], bond["coupon"], bond["ytm"], bond["maturity"])
    D_mod  = modified_duration(bond["face"], bond["coupon"], bond["ytm"], bond["maturity"])
    conv   = calc_convexity(bond["face"], bond["coupon"], bond["ytm"], bond["maturity"])
    mktval = price / 100 * bond["notional"]
    pos_dv01 = dv01_ / 100 * bond["notional"]
    total_dv01 += pos_dv01

    rows.append({
        "Bond": bond["name"],
        "YTM (%)": round(bond["ytm"] * 100, 3),
        "Price": round(price, 4),
        "Mod Duration": round(D_mod, 4),
        "DV01 (per 100)": round(dv01_, 4),
        "Notional": bond["notional"],
        "Market Value": round(mktval, 2),
        "Position DV01": round(pos_dv01, 2),
        "Convexity": round(conv, 4),
    })

portfolio_df = pd.DataFrame(rows)
display(portfolio_df)
print(f"\n✅ Portfolio total DV01: £{total_dv01:,.2f} per bp")
print(f"   (This means: if all rates rise 1bp, portfolio loses ~£{total_dv01:,.0f})")


,Bond,YTM (%),Price,Mod Duration,DV01 (per 100),Notional,Market Value,Position DV01,Convexity
0,2Y Bond,4.2000,100.0000,1.8807,0.0188,5000000,5000000.0000,940.3500,5.3776
1,5Y Bond,4.2700,100.0000,4.4183,0.0442,5000000,5000000.0000,2209.1300,24.7158
2,10Y Bond,4.4900,100.0000,7.9166,0.0792,5000000,5000000.0000,3958.3100,77.8678



✅ Portfolio total DV01: £7,107.78 per bp
   (This means: if all rates rise 1bp, portfolio loses ~£7,108)


In [14]:
# CELL 11: Scenario engine — parallel yield shifts

def scenario_pnl(portfolio_list, shift_bps):
    """
    Approximate P&L for a parallel shift using duration + convexity.
    Returns total P&L across the portfolio.
    """
    shift = shift_bps / 10_000  # convert bps to decimal
    total_pnl = 0.0
    for bond in portfolio_list:
        price  = price_bond(bond["face"], bond["coupon"], bond["ytm"], bond["maturity"])
        D_mod  = modified_duration(bond["face"], bond["coupon"], bond["ytm"], bond["maturity"])
        conv   = calc_convexity(bond["face"], bond["coupon"], bond["ytm"], bond["maturity"])
        # Duration + convexity approximation
        dP = -D_mod * price * shift + 0.5 * conv * price * shift**2
        pnl = dP / 100 * bond["notional"]
        total_pnl += pnl
    return total_pnl

# Test across a range of shifts: -200bps to +200bps
shifts = list(range(-200, 210, 25))
pnls   = [scenario_pnl(portfolio, s) for s in shifts]

# Chart
fig = px.line(
    x=shifts, y=pnls,
    labels={"x": "Parallel Shift (bps)", "y": "Portfolio P&L (£)"},
    title="Portfolio P&L vs Parallel Rate Shift (Duration + Convexity Approx)",
    markers=True,
)
fig.add_hline(y=0, line_dash="dash", line_color="red")
fig.update_layout(template="plotly_white")
fig.show()

print("✅ Scenario chart displayed")
print(f"   Example: +100bp shift → P&L = £{scenario_pnl(portfolio, 100):,.0f}")
print(f"   Example: -100bp shift → P&L = £{scenario_pnl(portfolio, -100):,.0f}")


✅ Scenario chart displayed
   Example: +100bp shift → P&L = £-683,788
   Example: -100bp shift → P&L = £737,769


In [15]:
# CELL 12: DV01-neutral hedge using IEF and comparison chart

# Step 1: ETF DV01 per £1 of price
ief_price = latest_etf_prices.get("IEF", 95.0)
ief_dur   = ETF_APPROX_DURATIONS["IEF"]
etf_dv01_per_unit = ief_dur * ief_price * 1e-4  # DV01 per share

print(f"IEF price:            £{ief_price:.2f}")
print(f"IEF eff. duration:    {ief_dur:.1f} years")
print(f"IEF DV01 per share:   £{etf_dv01_per_unit:.4f}")

# Step 2: How many shares to short?
# We need: hedge_shares * etf_dv01 + portfolio_dv01 = 0
hedge_shares = -total_dv01 / etf_dv01_per_unit
print(f"\nPortfolio DV01:       £{total_dv01:,.2f}")
print(f"Required hedge:       Short {hedge_shares:,.0f} shares of IEF")

# Step 3: P&L of hedge position under parallel shifts
def hedge_pnl(shift_bps, shares, etf_price, etf_dur):
    shift = shift_bps / 10_000
    dP_etf = -etf_dur * etf_price * shift  # approximate
    return dP_etf * shares  # negative shares = short

pnl_unhedged = [scenario_pnl(portfolio, s) for s in shifts]
pnl_hedge    = [hedge_pnl(s, hedge_shares, ief_price, ief_dur) for s in shifts]
pnl_hedged   = [u + h for u, h in zip(pnl_unhedged, pnl_hedge)]

# Chart comparison
fig = go.Figure()
fig.add_trace(go.Scatter(x=shifts, y=pnl_unhedged, name="Unhedged", mode="lines+markers"))
fig.add_trace(go.Scatter(x=shifts, y=pnl_hedged,   name="Hedged (IEF)", mode="lines+markers"))
fig.add_hline(y=0, line_dash="dash", line_color="grey")
fig.update_layout(
    title="Hedged vs Unhedged Portfolio P&L — Parallel Shifts",
    xaxis_title="Parallel Shift (bps)",
    yaxis_title="Approximate P&L (£)",
    template="plotly_white"
)
fig.show()

# Hedge efficiency metric
import numpy as np
hedge_efficiency = 1 - np.std(pnl_hedged) / np.std(pnl_unhedged)
print(f"\n✅ Hedge efficiency: {hedge_efficiency:.1%}")
print("   (higher = better hedge. 90%+ is strong for a single-factor hedge)")


IEF price:            £94.02
IEF eff. duration:    7.8 years
IEF DV01 per share:   £0.0733

Portfolio DV01:       £7,107.78
Required hedge:       Short -96,921 shares of IEF



✅ Hedge efficiency: 95.9%
   (higher = better hedge. 90%+ is strong for a single-factor hedge)


In [16]:
# CELL 13: Export results to Excel

output_path = "treasury_hedging_workbench.xlsx"

with pd.ExcelWriter(output_path, engine="openpyxl") as writer:
    portfolio_df.to_excel(writer, sheet_name="Portfolio", index=False)
    zero_curve_df.to_excel(writer, sheet_name="Zero Curve", index=False)
    curve_df.tail(252).mul(100).round(4).to_excel(writer, sheet_name="Yield History")

print(f"✅ Excel file saved: {output_path}")
print("   To download: Files icon (left sidebar) → right-click → Download")


✅ Excel file saved: treasury_hedging_workbench.xlsx
   To download: Files icon (left sidebar) → right-click → Download
